# Final Project — Telecom: Identifying Inefficient Operators

## Business Context

The virtual phone service **CallMeMaybe** is developing a new feature to provide supervisors with insights on underperforming operators. CallMeMaybe's clients are organizations that distribute incoming calls among multiple operators or make outgoing calls through them.

## Inefficiency Criteria

An operator is considered **inefficient** si cumple con alguna de las siguientes condiciones:
1. **High rate of missed incoming calls** (internal and external)
2. **Long wait times** on incoming calls
3. **Low number of outgoing calls** (if the operator has outbound duties)

## Objectives
1. Perform exploratory data analysis (EDA)
2. Identify inefficient operators with quantitative criteria
3. Test statistical hypotheses

## 1. Data Loading and Initial Exploration

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficos
plt.style.use('seaborn-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# Cargar datasets
telecom = pd.read_csv('/datasets/telecom_dataset_new.csv')
clients = pd.read_csv('/datasets/telecom_clients.csv')

print(f'Dataset de llamadas: {telecom.shape[0]:,} filas x {telecom.shape[1]} columnas')
print(f'Dataset de clientes: {clients.shape[0]:,} filas x {clients.shape[1]} columnas')

### 1.1 Call Dataset Exploration

In [ ]:
print('Información general:')
telecom.info()
print('\nPrimeras filas:')
telecom.head()

In [ ]:
print('Estadísticas descriptivas:')
telecom.describe()

In [ ]:
print('Valores nulos por columna:')
print(telecom.isnull().sum())
print(f'\nPorcentaje de nulos en operator_id: {telecom["operator_id"].isnull().mean()*100:.1f}%')
print(f'Porcentaje de nulos en internal: {telecom["internal"].isnull().mean()*100:.1f}%')

In [ ]:
print(f'Duplicados exactos: {telecom.duplicated().sum()}')
print(f'Porcentaje: {telecom.duplicated().mean()*100:.1f}%')

In [ ]:
print('Distribución de valores categóricos:')
print(f'\nDirección de llamada:')
print(telecom['direction'].value_counts())
print(f'\nTipo de llamada (interna/externa):')
print(telecom['internal'].value_counts(dropna=False))
print(f'\nLlamadas perdidas:')
print(telecom['is_missed_call'].value_counts())

### 1.2 Null Value Analysis in operator_id

In [ ]:
# Investigar los nulos en operator_id
null_ops = telecom[telecom['operator_id'].isna()]

print('Registros sin operator_id:')
print(f'  Total: {len(null_ops):,}')
print(f'\n  Por dirección:')
print(null_ops['direction'].value_counts().to_string())
print(f'\n  Por is_missed_call:')
print(null_ops['is_missed_call'].value_counts().to_string())
print(f'\n  Porcentaje de llamadas perdidas entre nulos: {null_ops["is_missed_call"].mean()*100:.1f}%')

print('\nInterpretación: La gran mayoría de los registros sin operator_id son llamadas')
print('entrantes perdidas (98.5%). Esto indica que son llamadas que ingresaron al sistema')
print('pero nunca fueron asignadas a un operador, probablemente porque el cliente colgó')
print('antes de ser atendido.')

### 1.3 Client Dataset Exploration

In [ ]:
print('Información general:')
clients.info()
print('\nDistribución por plan tarifario:')
print(clients['tariff_plan'].value_counts())
print(f'\nClientes únicos: {clients["user_id"].nunique()}')
clients.head()

## 2. Data Preprocessing

### 2.1 Type Conversion and Cleaning

In [ ]:
# Convertir fechas
telecom['date'] = pd.to_datetime(telecom['date'])
clients['date_start'] = pd.to_datetime(clients['date_start'])

print(f'Rango de fechas en llamadas: {telecom["date"].min().date()} a {telecom["date"].max().date()}')
print(f'Rango de registro de clientes: {clients["date_start"].min().date()} a {clients["date_start"].max().date()}')

In [ ]:
# Eliminar duplicados
print(f'Filas antes de eliminar duplicados: {len(telecom):,}')
telecom = telecom.drop_duplicates()
print(f'Filas después de eliminar duplicados: {len(telecom):,}')
print(f'Duplicados eliminados: {53902 - len(telecom):,}')

In [ ]:
# Tratar nulos en 'internal'
# Los 117 nulos en internal son un porcentaje muy bajo (0.2%), los eliminamos
print(f'Filas con internal nulo: {telecom["internal"].isna().sum()}')
telecom = telecom.dropna(subset=['internal'])
print(f'Filas después de eliminar nulos en internal: {len(telecom):,}')

In [ ]:
# Crear columna de tiempo de espera
telecom['wait_time'] = telecom['total_call_duration'] - telecom['call_duration']

print('Estadísticas de wait_time (segundos):')
print(telecom['wait_time'].describe())
print(f'\nValores negativos de wait_time: {(telecom["wait_time"] < 0).sum()}')

In [ ]:
# Unir con datos de clientes
telecom_full = telecom.merge(clients, on='user_id', how='left')

print(f'Dataset combinado: {len(telecom_full):,} filas')
print(f'Usuarios sin plan tarifario: {telecom_full["tariff_plan"].isna().sum()}')

## 3. Exploratory Data Analysis (EDA)

### 3.1 General Call Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Dirección de llamadas
telecom['direction'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%',
                                         colors=['#3498db', '#e74c3c'], startangle=90)
axes[0].set_title('Llamadas por dirección')
axes[0].set_ylabel('')

# Internas vs Externas
telecom['internal'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                        colors=['#2ecc71', '#f39c12'], startangle=90,
                                        labels=['Externa', 'Interna'])
axes[1].set_title('Llamadas internas vs externas')
axes[1].set_ylabel('')

# Perdidas vs Contestadas
telecom['is_missed_call'].value_counts().plot(kind='pie', ax=axes[2], autopct='%1.1f%%',
                                              colors=['#2ecc71', '#e74c3c'], startangle=90,
                                              labels=['Contestada', 'Perdida'])
axes[2].set_title('Llamadas contestadas vs perdidas')
axes[2].set_ylabel('')

plt.suptitle('Distribución general de llamadas', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Call Duration and Wait Time Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de duración (limitado al percentil 95 para mejor visualización)
p95_dur = telecom['call_duration'].quantile(0.95)
telecom[telecom['call_duration'] <= p95_dur]['call_duration'].hist(
    bins=50, ax=axes[0], color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribución de duración de llamada')
axes[0].set_xlabel('Duración (seg)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(telecom['call_duration'].mean(), color='red', linestyle='--', label=f'Media: {telecom["call_duration"].mean():.0f}s')
axes[0].axvline(telecom['call_duration'].median(), color='green', linestyle='--', label=f'Mediana: {telecom["call_duration"].median():.0f}s')
axes[0].legend()

# Histograma de tiempo de espera
p95_wait = telecom['wait_time'].quantile(0.95)
telecom[telecom['wait_time'] <= p95_wait]['wait_time'].hist(
    bins=50, ax=axes[1], color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribución del tiempo de espera')
axes[1].set_xlabel('Tiempo de espera (seg)')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(telecom['wait_time'].mean(), color='red', linestyle='--', label=f'Media: {telecom["wait_time"].mean():.0f}s')
axes[1].axvline(telecom['wait_time'].median(), color='green', linestyle='--', label=f'Mediana: {telecom["wait_time"].median():.0f}s')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.3 Temporal Analysis: Calls Per Day

In [ ]:
# Llamadas por día
daily_calls = telecom.groupby(telecom['date'].dt.date)['calls_count'].sum().reset_index()
daily_calls.columns = ['date', 'calls']

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_calls['date'], daily_calls['calls'], marker='.', linewidth=1, color='steelblue')
ax.set_title('Número total de llamadas por día')
ax.set_xlabel('Fecha')
ax.set_ylabel('Cantidad de llamadas')
ax.axhline(daily_calls['calls'].mean(), color='red', linestyle='--', alpha=0.7, 
           label=f'Media: {daily_calls["calls"].mean():.0f}')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.4 Call Analysis by Direction and Type

In [ ]:
# Tasa de llamadas perdidas por dirección e tipo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por dirección
missed_by_dir = telecom.groupby('direction')['is_missed_call'].mean() * 100
missed_by_dir.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'], edgecolor='black')
axes[0].set_title('Tasa de llamadas perdidas por dirección')
axes[0].set_ylabel('Porcentaje (%)')
axes[0].set_xticklabels(['Entrante', 'Saliente'], rotation=0)
for i, v in enumerate(missed_by_dir):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=12)

# Por tipo (interna/externa)
missed_by_type = telecom.groupby('internal')['is_missed_call'].mean() * 100
missed_by_type.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#f39c12'], edgecolor='black')
axes[1].set_title('Tasa de llamadas perdidas por tipo')
axes[1].set_ylabel('Porcentaje (%)')
axes[1].set_xticklabels(['Externa', 'Interna'], rotation=0)
for i, v in enumerate(missed_by_type):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

### 3.5 Analysis by Tariff Plan

In [ ]:
# Métricas por plan tarifario
tariff_stats = telecom_full.groupby('tariff_plan').agg(
    total_calls=('calls_count', 'sum'),
    missed_rate=('is_missed_call', 'mean'),
    avg_duration=('call_duration', 'mean'),
    avg_wait=('wait_time', 'mean'),
    n_operators=('operator_id', 'nunique')
).round(2)

tariff_stats['missed_rate'] = (tariff_stats['missed_rate'] * 100).round(2)
print('Métricas por plan tarifario:')
tariff_stats

## 4. Identifying Inefficient Operators

To identify inefficient operators, calcularemos tres métricas clave por operador y usaremos los **percentiles 90** como umbrales para identificar outliers negativos.

### 4.1 Calculate Metrics Per Operator

In [ ]:
# Filtrar solo registros con operator_id
ops_data = telecom[telecom['operator_id'].notna()].copy()

# --- Métrica 1: Tasa de llamadas entrantes perdidas ---
incoming = ops_data[ops_data['direction'] == 'in']

missed_rate = incoming.groupby('operator_id').apply(
    lambda x: x[x['is_missed_call']]['calls_count'].sum() / x['calls_count'].sum()
    if x['calls_count'].sum() > 0 else 0
).reset_index(name='missed_call_rate')

# --- Métrica 2: Tiempo de espera promedio en llamadas entrantes ---
avg_wait = incoming.groupby('operator_id')['wait_time'].mean().reset_index(name='avg_wait_time')

# --- Métrica 3: Total de llamadas salientes ---
outgoing = ops_data[ops_data['direction'] == 'out']
out_calls = outgoing.groupby('operator_id')['calls_count'].sum().reset_index(name='total_outgoing')

# Combinar métricas
operator_metrics = missed_rate.merge(avg_wait, on='operator_id', how='outer')
operator_metrics = operator_metrics.merge(out_calls, on='operator_id', how='outer')
operator_metrics['total_outgoing'] = operator_metrics['total_outgoing'].fillna(0)

print(f'Total de operadores analizados: {len(operator_metrics)}')
print('\nEstadísticas de las métricas:')
operator_metrics.describe().round(4)

### 4.2 Define Inefficiency Thresholds

In [ ]:
# Umbrales basados en percentiles
# Operadores con tasa de perdidas ALTA (percentil 90)
missed_threshold = operator_metrics['missed_call_rate'].quantile(0.90)

# Operadores con tiempo de espera ALTO (percentil 90)
wait_threshold = operator_metrics['avg_wait_time'].quantile(0.90)

# Operadores con pocas llamadas salientes (percentil 10, solo entre los que hacen llamadas salientes)
out_operators = operator_metrics[operator_metrics['total_outgoing'] > 0]
outgoing_threshold = out_operators['total_outgoing'].quantile(0.10)

print('UMBRALES DE INEFICACIA')
print('=' * 50)
print(f'Tasa de llamadas perdidas > {missed_threshold:.4f} ({missed_threshold*100:.2f}%)')
print(f'Tiempo de espera promedio > {wait_threshold:.2f} segundos')
print(f'Llamadas salientes < {outgoing_threshold:.0f} (solo operadores outbound)')

In [ ]:
# Identificar operadores ineficaces
# Criterio 1: Alta tasa de llamadas perdidas
high_missed = operator_metrics[operator_metrics['missed_call_rate'] > missed_threshold]['operator_id']

# Criterio 2: Tiempo de espera prolongado
high_wait = operator_metrics[operator_metrics['avg_wait_time'] > wait_threshold]['operator_id']

# Criterio 3: Pocas llamadas salientes (solo para operadores que hacen outbound)
low_outgoing = out_operators[out_operators['total_outgoing'] < outgoing_threshold]['operator_id']

# Operadores ineficaces: cumplen al menos 1 criterio
inefficient_ids = set(high_missed) | set(high_wait) | set(low_outgoing)

operator_metrics['is_inefficient'] = operator_metrics['operator_id'].isin(inefficient_ids)
operator_metrics['high_missed'] = operator_metrics['operator_id'].isin(high_missed)
operator_metrics['high_wait'] = operator_metrics['operator_id'].isin(high_wait)
operator_metrics['low_outgoing'] = operator_metrics['operator_id'].isin(low_outgoing)

print(f'OPERADORES INEFICACES IDENTIFICADOS')
print(f'=' * 50)
print(f'Por alta tasa de llamadas perdidas: {len(high_missed)}')
print(f'Por tiempo de espera prolongado: {len(high_wait)}')
print(f'Por pocas llamadas salientes: {len(low_outgoing)}')
print(f'\nTotal de operadores ineficaces (al menos 1 criterio): {len(inefficient_ids)}')
print(f'Total de operadores analizados: {len(operator_metrics)}')
print(f'Porcentaje ineficaces: {len(inefficient_ids)/len(operator_metrics)*100:.1f}%')

### 4.3 Inefficient Operators Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribución de tasa de llamadas perdidas
axes[0].hist(operator_metrics['missed_call_rate'], bins=50, color='steelblue', 
             edgecolor='black', alpha=0.7)
axes[0].axvline(missed_threshold, color='red', linestyle='--', linewidth=2,
                label=f'Umbral P90: {missed_threshold:.4f}')
axes[0].set_title('Tasa de llamadas perdidas')
axes[0].set_xlabel('Tasa')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Distribución de tiempo de espera
p95 = operator_metrics['avg_wait_time'].quantile(0.95)
axes[1].hist(operator_metrics[operator_metrics['avg_wait_time'] <= p95]['avg_wait_time'], 
             bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].axvline(wait_threshold, color='red', linestyle='--', linewidth=2,
                label=f'Umbral P90: {wait_threshold:.0f}s')
axes[1].set_title('Tiempo de espera promedio')
axes[1].set_xlabel('Segundos')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

# Distribución de llamadas salientes
p95_out = out_operators['total_outgoing'].quantile(0.95)
axes[2].hist(out_operators[out_operators['total_outgoing'] <= p95_out]['total_outgoing'], 
             bins=50, color='#2ecc71', edgecolor='black', alpha=0.7)
axes[2].axvline(outgoing_threshold, color='red', linestyle='--', linewidth=2,
                label=f'Umbral P10: {outgoing_threshold:.0f}')
axes[2].set_title('Total de llamadas salientes')
axes[2].set_xlabel('Cantidad')
axes[2].set_ylabel('Frecuencia')
axes[2].legend()

plt.suptitle('Distribución de métricas y umbrales de ineficacia', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Tasa de pérdidas vs Tiempo de espera
fig, ax = plt.subplots(figsize=(10, 7))

efficient = operator_metrics[~operator_metrics['is_inefficient']]
inefficient = operator_metrics[operator_metrics['is_inefficient']]

ax.scatter(efficient['missed_call_rate'], efficient['avg_wait_time'], 
           alpha=0.5, label=f'Eficaces ({len(efficient)})', color='steelblue', s=30)
ax.scatter(inefficient['missed_call_rate'], inefficient['avg_wait_time'], 
           alpha=0.7, label=f'Ineficaces ({len(inefficient)})', color='red', s=50, marker='x')

ax.axvline(missed_threshold, color='red', linestyle='--', alpha=0.5, label=f'Umbral pérdidas: {missed_threshold:.4f}')
ax.axhline(wait_threshold, color='orange', linestyle='--', alpha=0.5, label=f'Umbral espera: {wait_threshold:.0f}s')

ax.set_xlabel('Tasa de llamadas perdidas')
ax.set_ylabel('Tiempo de espera promedio (seg)')
ax.set_title('Operadores: Tasa de pérdidas vs Tiempo de espera')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### 4.4 Inefficient Operators Profile

In [ ]:
# Comparación de métricas: eficaces vs ineficaces
comparison = operator_metrics.groupby('is_inefficient').agg(
    n_operators=('operator_id', 'count'),
    avg_missed_rate=('missed_call_rate', 'mean'),
    avg_wait_time=('avg_wait_time', 'mean'),
    avg_outgoing=('total_outgoing', 'mean')
).round(4)

comparison.index = ['Eficaces', 'Ineficaces']
comparison['avg_missed_rate'] = (comparison['avg_missed_rate'] * 100).round(2)

print('Comparación de métricas promedio:')
comparison

## 5. Statistical Hypothesis Testing

### Hypothesis 1: Missed Call Rate

**H₀:** La tasa de llamadas perdidas de los operadores ineficaces NO es significativamente diferente de la de los operadores eficaces.  
**H₁:** La tasa de llamadas perdidas de los operadores ineficaces ES significativamente mayor.  
**α = 0.05**

In [ ]:
alpha = 0.05

group_eff = operator_metrics[~operator_metrics['is_inefficient']]['missed_call_rate']
group_ineff = operator_metrics[operator_metrics['is_inefficient']]['missed_call_rate']

# Prueba de normalidad
if len(group_eff) > 5000:
    stat_e, p_norm_e = stats.normaltest(group_eff)
    stat_i, p_norm_i = stats.normaltest(group_ineff)
else:
    stat_e, p_norm_e = stats.shapiro(group_eff[:5000])
    stat_i, p_norm_i = stats.shapiro(group_ineff[:5000])

print('Prueba de normalidad (Shapiro-Wilk):')
print(f'  Eficaces: p-value = {p_norm_e:.6f} → {"Normal" if p_norm_e > 0.05 else "No normal"}')
print(f'  Ineficaces: p-value = {p_norm_i:.6f} → {"Normal" if p_norm_i > 0.05 else "No normal"}')
print()

# Como no son normales, usamos Mann-Whitney U
stat, p_value = stats.mannwhitneyu(group_ineff, group_eff, alternative='greater')
print('Prueba de Mann-Whitney U (unilateral):')
print(f'  Estadístico U: {stat:.2f}')
print(f'  p-value: {p_value:.6f}')
print(f'  Media eficaces: {group_eff.mean()*100:.4f}%')
print(f'  Media ineficaces: {group_ineff.mean()*100:.4f}%')
print()
if p_value < alpha:
    print(f'  → RECHAZAMOS H₀ (p = {p_value:.6f} < α = {alpha})')
    print('  → Los operadores ineficaces tienen una tasa de llamadas perdidas significativamente MAYOR.')
else:
    print(f'  → NO rechazamos H₀ (p = {p_value:.6f} >= α = {alpha})')

### Hypothesis 2: Wait Time

**H₀:** El tiempo de espera promedio de los operadores ineficaces NO es significativamente diferente del de los operadores eficaces.  
**H₁:** El tiempo de espera promedio de los operadores ineficaces ES significativamente mayor.  
**α = 0.05**

In [ ]:
group_eff_wait = operator_metrics[~operator_metrics['is_inefficient']]['avg_wait_time'].dropna()
group_ineff_wait = operator_metrics[operator_metrics['is_inefficient']]['avg_wait_time'].dropna()

# Mann-Whitney U (distribuciones no normales)
stat, p_value = stats.mannwhitneyu(group_ineff_wait, group_eff_wait, alternative='greater')

print('Prueba de Mann-Whitney U (unilateral):')
print(f'  Estadístico U: {stat:.2f}')
print(f'  p-value: {p_value:.6f}')
print(f'  Media eficaces: {group_eff_wait.mean():.2f} seg')
print(f'  Media ineficaces: {group_ineff_wait.mean():.2f} seg')
print()
if p_value < alpha:
    print(f'  → RECHAZAMOS H₀ (p = {p_value:.6f} < α = {alpha})')
    print('  → Los operadores ineficaces tienen un tiempo de espera significativamente MAYOR.')
else:
    print(f'  → NO rechazamos H₀ (p = {p_value:.6f} >= α = {alpha})')

### Hypothesis 3: Differences by Tariff Plan

**H₀:** No hay diferencia significativa en la tasa de llamadas perdidas entre los diferentes planes tarifarios.  
**H₁:** Existe al menos un plan tarifario con una tasa de llamadas perdidas significativamente diferente.  
**α = 0.05**

In [ ]:
# Calcular tasa de llamadas perdidas por operador y plan tarifario
ops_with_tariff = ops_data.merge(clients[['user_id', 'tariff_plan']], on='user_id', how='inner')
incoming_tariff = ops_with_tariff[ops_with_tariff['direction'] == 'in']

# Tasa de pérdidas por plan
tariff_groups = {}
for plan in clients['tariff_plan'].unique():
    plan_data = incoming_tariff[incoming_tariff['tariff_plan'] == plan]
    plan_rates = plan_data.groupby('operator_id').apply(
        lambda x: x[x['is_missed_call']]['calls_count'].sum() / x['calls_count'].sum()
        if x['calls_count'].sum() > 0 else 0
    )
    tariff_groups[plan] = plan_rates
    print(f'Plan {plan}: {len(plan_rates)} operadores, tasa promedio de pérdidas: {plan_rates.mean()*100:.2f}%')

# Prueba de Kruskal-Wallis (alternativa no paramétrica a ANOVA)
groups = [g.values for g in tariff_groups.values() if len(g) > 0]
stat, p_value = stats.kruskal(*groups)

print(f'\nPrueba de Kruskal-Wallis:')
print(f'  Estadístico H: {stat:.4f}')
print(f'  p-value: {p_value:.6f}')
print()
if p_value < alpha:
    print(f'  → RECHAZAMOS H₀ (p = {p_value:.6f} < α = {alpha})')
    print('  → Existe al menos un plan con tasa de pérdidas significativamente diferente.')
else:
    print(f'  → NO rechazamos H₀ (p = {p_value:.6f} >= α = {alpha})')
    print('  → No hay diferencia significativa entre planes tarifarios.')

## 6. Conclusions and Recommendations

### Key EDA Findings

1. **Composición de llamadas:** Las llamadas salientes representan la mayoría del tráfico (~60%), mientras que las internas son solo ~11%. La tasa general de llamadas perdidas es significativa y merece atención de los supervisores.

2. **Valores nulos en operator_id:** Los 8,172 registros sin operador asignado corresponden casi en su totalidad a llamadas entrantes perdidas (98.5%). Esto representa llamadas donde el cliente colgó antes de ser conectado con un operador, lo cual es un indicador de problemas en la distribución de llamadas.

3. **Duplicados:** Se eliminaron 4,900 registros duplicados exactos (9.1%), lo que sugiere posibles problemas en el proceso de registro de datos.

4. **Tiempo de espera:** La distribución del tiempo de espera es fuertemente sesgada a la derecha, con una mediana de 55 segundos pero casos extremos de más de 46,000 segundos.

### Inefficient Operators Identified

Se identificaron operadores ineficaces utilizando tres criterios basados en percentiles:
- **Alta tasa de llamadas perdidas** (por encima del percentil 90)
- **Long wait times** (por encima del percentil 90)
- **Low number of outgoing calls** (por debajo del percentil 10, solo para operadores outbound)

### Hypothesis Testing Results

Las pruebas estadísticas confirman que las diferencias entre operadores eficaces e ineficaces son estadísticamente significativas, validando los criterios de clasificación utilizados.

### Recommendations for Supervisors

1. **Monitoreo prioritario:** Implementar un sistema de alertas tempranas para operadores que superen los umbrales de ineficacia definidos.
2. **Capacitación:** Los operadores con alta tasa de llamadas perdidas y tiempos de espera elevados podrían beneficiarse de entrenamiento adicional en gestión de llamadas.
3. **Redistribución de carga:** Considerar balancear la distribución de llamadas entrantes para reducir los tiempos de espera.
4. **Seguimiento a llamadas sin operador:** Las 8,172 llamadas sin operador asignado representan una pérdida de servicio significativa; se recomienda revisar el algoritmo de enrutamiento de llamadas.
5. **Revisión periódica:** Los umbrales de ineficacia deben recalcularse periódicamente a medida que el servicio mejore.

## 7. Sources Consulted

1. **Documentación de pandas** (https://pandas.pydata.org/docs/) — Referencia para manipulación de DataFrames, merge, groupby y tratamiento de valores nulos.

2. **Documentación de SciPy - Stats** (https://docs.scipy.org/doc/scipy/reference/stats.html) — Guía para selección e implementación de pruebas estadísticas (Mann-Whitney U, Kruskal-Wallis, Shapiro-Wilk).

3. **Documentación de Matplotlib** (https://matplotlib.org/stable/gallery/index.html) — Referencia para creación de histogramas, gráficos circulares y scatter plots.

4. **Documentación de Seaborn** (https://seaborn.pydata.org/) — Referencia para visualizaciones estadísticas y configuración de estilos.

5. **"Practical Statistics for Data Scientists" — Bruce & Bruce** — Guía conceptual para la selección de pruebas estadísticas paramétricas vs no paramétricas y el uso de percentiles como umbrales.

6. **Stack Overflow** (https://stackoverflow.com/) — Resolución de dudas específicas sobre agrupación condicional en pandas y manejo de valores nulos.

7. **"Call Center Metrics: Best Practices" — ICMI** (https://www.icmi.com/) — Referencia sobre métricas estándar de la industria de call centers (tasa de abandono, tiempo de espera, nivel de servicio) que orientaron la definición de criterios de ineficacia.